# ECE 343 Lab: Electrostatic Fields in Python (Lesson 26)

<a href="https://colab.research.google.com/github/livethisdream/ece343/blob/main/book/extras/labs/ECE343_Lab_L26_Electrostatics.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Team members:**

**Documentation:**

This lab computes the fields of Module 2 on a grid and checks them against the
results you derived by hand. Part A builds fields from Coulomb's law by
superposition (Lessons 18 and 19) and tests Gauss's law in both forms
(Lessons 20 and 21). Part B solves Laplace's equation for a coaxial cable and
finds its capacitance per meter (Lessons 23 and 25). Part C, if your
instructor assigns it, measures that capacitance on the bench.

**Teams.** Work in a team of two or three. Each team submits one report.

**Deliverable.** This notebook, run top to bottom with every code cell
complete and every **Answer** cell filled in, exported to PDF and uploaded to
Gradescope. Put every member's name in the cell above, and write a
Documentation line (**None** if you did not collaborate outside your team).

**Running it.** Open it in Google Colab with the badge above, or run it
locally with Python 3, NumPy, and Matplotlib. Run the cells in order. A cell
marked `TODO` is yours to finish; the rest are given.

**Reference.** Ellingson, *Electromagnetics*, Vol. 1, Chapter 5, and the
Lesson 18 through 25 pages on the course site.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

EPS0 = 8.854e-12                  # permittivity of free space (F/m)
K = 1 / (4 * np.pi * EPS0)        # Coulomb constant, about 8.988e9 (m/F)
nC = 1e-9                         # one nanocoulomb (C)
trapezoid = getattr(np, "trapezoid", None) or np.trapz   # NumPy 2 renamed trapz


def plot_field(X, Y, Ex, Ey, title, charges=()):
    """Field lines colored by log10 |E|, with the point charges marked."""
    mag = np.hypot(Ex, Ey)
    fig, ax = plt.subplots(figsize=(6, 6))
    strm = ax.streamplot(X, Y, Ex, Ey, color=np.log10(mag), cmap="viridis",
                         density=1.4, linewidth=1)
    fig.colorbar(strm.lines, ax=ax, label="log10 |E|  (E in V/m)", shrink=0.8)
    for xq, yq, q in charges:
        ax.plot(xq, yq, "o", color="tab:red" if q > 0 else "tab:blue", ms=4)
    ax.set_aspect("equal")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_title(title)
    plt.show()


def plot_scalar(X, Y, F, title, label, limit):
    """A heat map of F, clipped to +/- limit so a singularity does not wash it out."""
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.pcolormesh(X, Y, F, cmap="RdBu_r", vmin=-limit, vmax=limit, shading="auto")
    fig.colorbar(im, ax=ax, label=label)
    ax.set_aspect("equal")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_title(title)
    plt.show()


# The grid for Part A: 200 points a side from -0.5 m to 0.5 m. An even count
# keeps every grid point off the origin, where a charge sits.
x = np.linspace(-0.5, 0.5, 200)
h = x[1] - x[0]
X, Y = np.meshgrid(x, x)

---
## Part A: Coulomb's Law and Gauss's Law

### A1. The field of point charges

Coulomb's law gives the field of a point charge $q$ at $\mathbf{r}'$ as

$$\mathbf{E}(\mathbf{r}) = \frac{q}{4\pi\epsilon_0}\,\frac{\mathbf{r} - \mathbf{r}'}{\vert\mathbf{r} - \mathbf{r}'\vert^3},$$

and superposition adds the fields of several charges (Lesson 18). Every
charge in Part A sits in the $z = 0$ plane, and we evaluate the field in that
plane, where $E_z = 0$ by symmetry.

The function below takes whole arrays of points, so one call fills the grid;
there is no loop over grid points. Finish it.

In [ ]:
def point_charge_field(X, Y, charges):
    """E (V/m) at the points (X, Y) in the z = 0 plane, from point charges.

    charges is a list of (x, y, q), with x and y in meters and q in coulombs.
    """
    X, Y = np.broadcast_arrays(np.asarray(X, float), np.asarray(Y, float))
    Ex = np.zeros_like(X)
    Ey = np.zeros_like(X)
    for xq, yq, q in charges:
        rx, ry = X - xq, Y - yq            # vector from the charge to each point
        # TODO: add this charge's contribution to Ex and Ey (Coulomb's law).
        pass
    return Ex, Ey

In [ ]:
charges = [(0.0, 0.0, 1 * nC)]            # 1 nC at the origin
Ex, Ey = point_charge_field(X, Y, charges)
plot_field(X, Y, Ex, Ey, "A 1 nC point charge", charges)

Ex1, Ey1 = point_charge_field(0.1, 0.0, charges)
print(f"E at (0.1, 0) m:  Ex = {Ex1:.1f} V/m,  Ey = {Ey1:.1f} V/m")

**Question A1.** Work $\vert\mathbf{E}\vert$ at $0.1\ \text{m}$ from the
charge by hand. Does the code agree? Do the field lines point the way you
expect, and how does the color change with distance?

**Answer:**

### A2. A dipole

Place $+1\ \text{nC}$ at $(-0.1, 0)\ \text{m}$ and $-1\ \text{nC}$ at
$(+0.1, 0)\ \text{m}$.

In [ ]:
charges = []        # TODO: the two charges of the dipole, as (x, y, q) tuples
Ex, Ey = point_charge_field(X, Y, charges)
plot_field(X, Y, Ex, Ey, "A dipole", charges)

Ex1, Ey1 = point_charge_field(0.0, 0.3, charges)
print(f"E at (0, 0.3) m:  Ex = {Ex1:.2f} V/m,  Ey = {Ey1:.2f} V/m")

**Question A2.** On the bisector, Lesson 18 found the dipole's field by
superposition. Work $\mathbf{E}$ at $(0, 0.3)\ \text{m}$ by hand and compare.
Why does the $y$ component cancel there, and which way does the field point?

**Answer:**

### A3. A line of charge

A line charge is many point charges in a row (Lesson 18). Build a line of
length $L = 0.8\ \text{m}$ on the $x$ axis, centered on the origin, with
$\rho_\ell = 12.5\ \text{nC/m}$, out of $n = 80$ point charges: split the
line into $n$ pieces of length $\Delta x = L/n$ and put a charge
$\rho_\ell\,\Delta x$ at the center of each piece.

In [ ]:
def line_of_charges(x0, x1, y, rho_l, n=80):
    """A finite line of charge from (x0, y) to (x1, y), as n point charges."""
    # TODO: return a list of n (x, y, q) tuples.
    return []


rho_l = 12.5 * nC                          # C/m
charges = line_of_charges(-0.4, 0.4, 0.0, rho_l)
Ex, Ey = point_charge_field(X, Y, charges)
plot_field(X, Y, Ex, Ey, "A 0.8 m line of charge", charges)

Lesson 18 found the field on the perpendicular bisector of a finite line, and
its limit for an infinitely long line:

$$E_y = \frac{\rho_\ell}{2\pi\epsilon_0 d}\,\frac{L/2}{\sqrt{d^2 + (L/2)^2}}, \qquad
E_y \to \frac{\rho_\ell}{2\pi\epsilon_0 d} \text{ as } L \to \infty.$$

Fill in both and compare them with the code at several distances $d$.

In [ ]:
L = 0.8
print("   d (m)   code (V/m)   finite (V/m)   infinite (V/m)")
for d in (0.05, 0.1, 0.2, 0.4):
    _, E_code = point_charge_field(0.0, d, charges)
    E_finite = 0.0      # TODO: Lesson 18's finite-line result
    E_inf = 0.0         # TODO: the infinite-line limit
    print(f"{d:8.2f} {E_code:12.0f} {E_finite:14.0f} {E_inf:16.0f}")

**Question A3.** Where does the infinite-line approximation hold, and where does
it break down? Look at the table and at the field lines near the ends of the
line. What would make the code disagree with the finite-line formula?

**Answer:**

### A4. Two lines of opposite charge

Put $+12.5\ \text{nC/m}$ on a line at $y = +0.1\ \text{m}$ and
$-12.5\ \text{nC/m}$ on a line at $y = -0.1\ \text{m}$, both from
$x = -0.4$ to $0.4\ \text{m}$. This is the cross section of a strip capacitor,
the 2D cousin of Lesson 25's parallel plates.

In [ ]:
charges = []        # TODO: both lines, using line_of_charges
Ex, Ey = point_charge_field(X, Y, charges)
plot_field(X, Y, Ex, Ey, "Two lines of opposite charge", charges)

_, E_mid = point_charge_field(0.0, 0.0, charges)
print(f"Ey at the origin: {E_mid:.0f} V/m")

**Question A4.**
1. Predict $E_y$ at the origin from A3's results, and compare with the code.
2. Where is the field between the lines nearly uniform, and where does it fringe?
3. Change the bottom line to $+12.5\ \text{nC/m}$ and rerun. Where is $\mathbf{E} = 0$ now, and why?

**Answer:**

### A5. A charge distribution of your own

Build something interesting: a ring of charges, a square, a quadrupole, a
charge inside a ring of opposite charge. Plot it, then describe what you built
and explain one feature of the field from Lessons 18 through 22.

In [ ]:
charges = []        # TODO: your distribution
Ex, Ey = point_charge_field(X, Y, charges)
plot_field(X, Y, Ex, Ey, "Our distribution", charges)

**Answer:**

### A6. Gauss's law in point form, and a trap

Lesson 21 derived the point form of Gauss's law. In free space, away from any
charge,

$$\nabla\cdot\mathbf{E} = \frac{\partial E_x}{\partial x} + \frac{\partial E_y}{\partial y} + \frac{\partial E_z}{\partial z} = \frac{\rho_v}{\epsilon_0} = 0.$$

Our grid lives in the $z = 0$ plane, so the code can take only the first two
derivatives. Write a function that does, with `np.gradient`, and apply it to
the single point charge of A1.

In [ ]:
def divergence_2d(Ex, Ey, h):
    """dEx/dx + dEy/dy on the grid. Rows of the grid run along y, columns along x."""
    # TODO: np.gradient(F, h, axis=1) is dF/dx; axis=0 is dF/dy.
    return np.zeros_like(Ex)


charges = [(0.0, 0.0, 1 * nC)]
Ex, Ey = point_charge_field(X, Y, charges)
div = divergence_2d(Ex, Ey, h)
plot_scalar(X, Y, div, "2D divergence of a point charge's field",
            "dEx/dx + dEy/dy  (V/m^2)", limit=5000)

i, j = np.argmin(np.abs(x)), np.argmin(np.abs(x - 0.2))   # the grid point nearest (0.2, 0)
print(f"2D divergence near ({x[j]:.3f}, {x[i]:.3f}) m: {div[i, j]:.0f} V/m^2")

**Question A6.** The heat map is not zero away from the charge, so the 2D
divergence says there is negative charge everywhere. Is Gauss's law wrong?

1. Show that $\dfrac{\partial E_x}{\partial x} + \dfrac{\partial E_y}{\partial y} = -\dfrac{Kq}{r^3}$ in the plane, and check it against the printed value.
2. Write $E_z$ for the point charge at a height $z$ above the plane, and find $\dfrac{\partial E_z}{\partial z}$ at $z = 0$.
3. Explain what the 2D grid cannot see, and what kind of source would make a 2D test of Gauss's law honest.

**Answer:**

### A7. Gauss's law with line charges

An infinite line charge along $z$ through $(x_\ell, y_\ell)$ has the field
(Lesson 20)

$$\mathbf{E} = \frac{\rho_\ell}{2\pi\epsilon_0}\,\frac{\boldsymbol{\rho}}{\vert\boldsymbol{\rho}\vert^2},
\qquad \boldsymbol{\rho} = (x - x_\ell)\,\hat{\mathbf{x}} + (y - y_\ell)\,\hat{\mathbf{y}},$$

which is the same in every plane of constant $z$, so $\partial E_z/\partial z = 0$
and the 2D divergence is the whole divergence. Finish the function, then
repeat A6 with a $1\ \text{nC/m}$ line through the origin.

In [ ]:
def line_charge_field(X, Y, lines):
    """E (V/m) from infinite line charges along z.

    lines is a list of (x, y, rho_l), with rho_l in C/m.
    """
    X, Y = np.broadcast_arrays(np.asarray(X, float), np.asarray(Y, float))
    Ex = np.zeros_like(X)
    Ey = np.zeros_like(X)
    for xl, yl, rho_l in lines:
        rx, ry = X - xl, Y - yl
        # TODO: add this line's contribution to Ex and Ey.
        pass
    return Ex, Ey


lines = [(0.0, 0.0, 1 * nC)]
Ex, Ey = line_charge_field(X, Y, lines)
div = divergence_2d(Ex, Ey, h)
plot_scalar(X, Y, div, "Divergence of a line charge's field",
            "dEx/dx + dEy/dy  (V/m^2)", limit=5000)
print(f"Divergence near (0.2, 0) m: {div[i, j]:.2f} V/m^2,"
      f"  while |E|/r there is {np.hypot(Ex[i, j], Ey[i, j]) / r:.0f} V/m^2")

Now the integral form. Per meter of length along $z$, Gauss's law says the
outward flux of $\mathbf{D} = \epsilon_0\mathbf{E}$ through any closed contour
equals the line charge it encloses (Lesson 20):

$$\oint \epsilon_0\,\mathbf{E}\cdot\hat{\mathbf{n}}\,d\ell = \rho_{\ell,\text{enc}}.$$

The function below integrates around a square, edge by edge. It is given;
use it.

In [ ]:
def flux_per_length(field, sources, xc, yc, s, n=4001):
    """Outward flux of eps0*E (C/m) through a square of half-width s centered at (xc, yc)."""
    t = np.linspace(-s, s, n)
    Ex, _ = field(xc + s, yc + t, sources)
    right = trapezoid(Ex, t)
    Ex, _ = field(xc - s, yc + t, sources)
    left = -trapezoid(Ex, t)
    _, Ey = field(xc + t, yc + s, sources)
    top = trapezoid(Ey, t)
    _, Ey = field(xc + t, yc - s, sources)
    bottom = -trapezoid(Ey, t)
    return EPS0 * (right + left + top + bottom)


lines = [(0.0, 0.0, +1 * nC), (0.3, 0.0, -1 * nC)]
for xc, yc, s, what in [(0.0, 0.0, 0.1, "around the + line"),
                        (0.0, 0.0, 0.05, "a smaller square around it"),
                        (0.15, 0.0, 0.25, "around both lines"),
                        (-0.3, 0.2, 0.05, "around neither")]:
    q = flux_per_length(line_charge_field, lines, xc, yc, s)
    print(f"{what:28s} flux = {q / nC:+.4f} nC/m")

**Question A7.**
1. How does the divergence of the line charge's field compare with A6's, away from the line?
2. Explain each of the four fluxes with Gauss's law. Why does the size of the square not matter?

**Answer:**

---
## Part B: Laplace's Equation and the Capacitance of a Coax

Lesson 23 derived Laplace's equation, $\nabla^2 V = 0$, for the space between
conductors, and showed that on a grid of spacing $h$ it says each point's
potential is the **average of its four neighbors**:

$$V_{i,j} = \tfrac{1}{4}\left(V_{i+1,j} + V_{i-1,j} + V_{i,j+1} + V_{i,j-1}\right).$$

Fix the potential on the conductors, start with a guess in between, and replace
every point with its neighbors' average, over and over. The guess relaxes to
the solution, and Lesson 23's uniqueness proof says it is the only one.

Here the conductors are RG-58's: an inner conductor of radius
$a = 0.405\ \text{mm}$ held at $V_0 = 1\ \text{V}$, an outer conductor of
radius $b = 1.475\ \text{mm}$ at $0\ \text{V}$, and polyethylene between them
with $\epsilon_r = 2.25$. The function updates half the points at a time,
like the red squares and then the black squares of a checkerboard, and
over-corrects each step by a factor $\omega$; this converges in a few hundred
sweeps where the plain average would take thousands. Finish the average.

In [ ]:
a, b, eps_r = 0.405e-3, 1.475e-3, 2.25          # RG-58 (m)


def relax_coax(a, b, h, V0=1.0, tol=1e-7, max_sweeps=20000):
    """Solve Laplace's equation between coaxial conductors on a square grid.

    Returns the grid coordinates x (m), the potential V (V), and the sweep count.
    """
    n = int(np.ceil(b / h)) + 2
    x = np.arange(-n, n + 1) * h
    Xg, Yg = np.meshgrid(x, x)
    R = np.hypot(Xg, Yg)
    inner, outer = R <= a, R >= b
    fixed = inner | outer
    V = np.where(inner, V0, 0.0)                  # the starting guess
    omega = 2 / (1 + np.sin(np.pi * h / (2 * b)))  # over-relaxation factor
    I, J = np.indices(V.shape)
    red = ((I + J) % 2 == 0) & ~fixed
    black = ((I + J) % 2 == 1) & ~fixed
    for sweep in range(1, max_sweeps + 1):
        change = 0.0
        for points in (red, black):
            # TODO: avg = the average of each point's four neighbors.
            # np.roll(V, 1, axis=0) shifts V by one row; the grid's edges are
            # all outer conductor, so the wraparound does no harm.
            avg = V
            step = avg[points] - V[points]
            V[points] += omega * step
            change = max(change, np.abs(step).max())
        if change < tol:
            break
    return x, V, sweep

### B1. The potential

Solve on a grid with $h = b/60$, plot the potential with its equipotentials,
and compare $V$ along the $+x$ axis with Lesson 23's result for the coax.

In [ ]:
hB = b / 60
xB, V, sweeps = relax_coax(a, b, hB)
print(f"converged in {sweeps} sweeps on a {V.shape[0]} x {V.shape[1]} grid")

XB, YB = np.meshgrid(xB * 1e3, xB * 1e3)          # mm, for plotting
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.pcolormesh(XB, YB, V, cmap="viridis", shading="auto")
ax.contour(XB, YB, V, levels=np.linspace(0.1, 0.9, 9), colors="white", linewidths=0.8)
fig.colorbar(im, ax=ax, label="V (volts)")
ax.set_aspect("equal")
ax.set_xlabel("x (mm)")
ax.set_ylabel("y (mm)")
ax.set_title("RG-58 cross section, relaxed")
plt.show()

mid = len(xB) // 2
rho = xB[mid:]                                    # the +x axis
V_axis = V[mid, mid:]
inside = (rho > a) & (rho < b)
V_exact = np.zeros_like(rho)    # TODO: Lesson 23's V(rho) between the conductors (use rho[inside])

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(rho * 1e3, V_axis, "o", ms=3, label="relaxation")
ax.plot(rho[inside] * 1e3, V_exact[inside], label="Lesson 23")
ax.set_xlabel("rho (mm)")
ax.set_ylabel("V (volts)")
ax.legend()
plt.show()

**Question B1.** Describe the equipotentials. Are they evenly spaced? Where does
the relaxed potential differ most from Lesson 23's, and why there?

**Answer:**

### B2. Capacitance per meter, by Gauss's law

To get $C'$ we need the charge per meter on the inner conductor when it sits at
$V_0$. Lesson 24's boundary condition and Lesson 20's Gauss's law give it from
the field: the flux of $\mathbf{D} = \epsilon\mathbf{E}$ out through any contour
around the inner conductor is $\rho_\ell$, and

$$C' = \frac{\rho_\ell}{V_0}.$$

The function below adds up $\epsilon\,\mathbf{E}\cdot\hat{\mathbf{n}}$ around a
square of half-width $s$ that runs along grid lines. Finish it: find
$\mathbf{E} = -\nabla V$ on the grid.

In [ ]:
def cprime_gauss(x, V, h, s, eps_r, V0=1.0):
    """C' (F/m) from the flux of eps*E through a square of half-width s around the center."""
    Ey, Ex = np.zeros_like(V), np.zeros_like(V)   # TODO: E = -grad V (np.gradient returns d/dy, then d/dx)
    c, k = len(x) // 2, int(round(s / h))
    span = slice(c - k, c + k + 1)
    flux = (trapezoid(Ex[span, c + k], dx=h) - trapezoid(Ex[span, c - k], dx=h)
            + trapezoid(Ey[c + k, span], dx=h) - trapezoid(Ey[c - k, span], dx=h))
    return eps_r * EPS0 * flux / V0


C_exact = 0.0      # TODO: Lesson 25's C' for a coax
print(f"Lesson 25: C' = {C_exact * 1e12:.2f} pF/m")
for s in (0.55e-3, 0.7e-3, 0.9e-3):
    print(f"square of half-width {s * 1e3:.2f} mm: C' = {cprime_gauss(xB, V, hB, s, eps_r) * 1e12:.2f} pF/m")

**Question B2.** The square must stay between the conductors: why must
$a < s < b/\sqrt{2}$? Why do the three squares give the same $C'$? How close is
the grid's $C'$ to Lesson 25's, and to the $96.8\ \text{pF/m}$ that Lesson 6
quoted for RG-58?

**Answer:**

### B3. How fine a grid?

The 2% in B2 is the grid's error, not Gauss's. Repeat the calculation with
$h = b/30$, $b/60$, and $b/120$, and look at how the error changes with $h$.
The finest grid takes a few seconds.

In [ ]:
print("   N = b/h   sweeps   C' (pF/m)   error")
results = {}
for N in (30, 60, 120):
    hN = b / N
    xN, VN, sweeps = relax_coax(a, b, hN)
    C = cprime_gauss(xN, VN, hN, 0.7e-3, eps_r)
    results[N] = C
    print(f"{N:10d} {sweeps:8d} {C * 1e12:11.2f} {100 * (C - C_exact) / C_exact:8.2f}%")

**Question B3.**
1. When $h$ halves, what happens to the error? What does that say about where the error comes from?
2. If the error is proportional to $h$, then $C'(h) \approx C' + \alpha h$. Combine the $h = b/60$ and $b/120$ results to cancel the error term, and compare with Lesson 25.
3. Each halving of $h$ costs about eight times the run time. Roughly what $h$, and what cost, would a 0.1% answer take without the trick in 2?

**Answer:**

---
## Part C (if assigned): Measuring $C'$ on the Bench

A short length of coax with its far end open is a capacitor: Lesson 7's input
impedance of an open line, $Z_\text{in} = -jZ_0\cot\beta\ell$, becomes
$1/(j\omega C'\ell)$ when $\beta\ell \ll 1$. At 2 MHz a 2 m cable is
$0.02\lambda$ long, short enough that the correction is under 1%.

1. Calibrate the NanoVNA from 50 kHz to 5 MHz with the SMA open, short, and load at the end of the SMA cable, as in Lesson 16.
2. Add the SMA-to-BNC adapter and a measured length of RG-58 (about 1 m), with its far end open. At 2 MHz, record the reactance $X$ from the marker.
3. Repeat with a second cable of a different length (about 2 m).
4. Each cable's capacitance is $C = -1/(\omega X)$. The adapter and connectors add the same few picofarads to both, so take the difference: $C' = (C_2 - C_1)/(\ell_2 - \ell_1)$.

In [ ]:
f = 2e6                          # Hz
l1, X1 = 1.0, 0.0                # TODO: cable 1 length (m) and measured reactance (ohms)
l2, X2 = 2.0, 0.0                # TODO: cable 2

w = 2 * np.pi * f
C1, C2 = -1 / (w * X1), -1 / (w * X2)
print(f"C1 = {C1 * 1e12:.1f} pF,  C2 = {C2 * 1e12:.1f} pF")
print(f"C' = {(C2 - C1) / (l2 - l1) * 1e12:.1f} pF/m  (Lesson 25: {C_exact * 1e12:.1f} pF/m)")

**Question C.** How close is your $C'$ to $96.8\ \text{pF/m}$? Name two
things that could account for the difference.

**Answer:**

---
## Report

Before you submit, run the whole notebook from the top (**Runtime**, then
**Run all** in Colab) so every output is current. Then:

1. Check that every `TODO` is done and every **Answer** cell is filled in.
2. Put every team member's name and your Documentation line in the first cell.
3. Export to PDF (**File**, then **Print**, then **Save as PDF**) and upload it to Gradescope as one team submission, adding every member.